In [2]:
import subprocess
import sys

for pkg in ["eurostat", "pandas"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

import eurostat
import pandas as pd


In [3]:
# --- Fetch and prepare tour_ce_omn12 (tourism: overnight stays) ---
tourism_raw = eurostat.get_data_df('tour_ce_omn12')

# Normalise month column and rename geo
tourism_raw["month"] = tourism_raw["month"].str.replace(r'M', "", regex=True)
tourism_raw = tourism_raw.rename(columns={"geo\\TIME_PERIOD": "GEO"})

# Melt to long format
tourism_long = tourism_raw.melt(
    id_vars=['freq', 'indic_to', 'c_resid', 'month', 'unit', 'GEO'],
    var_name='TIME_PERIOD',
    value_name='bookings'
)

# Keep annual totals (month == "TOTAL") for overnight stays (STY)
# c_resid = TOTAL means domestic + foreign combined
tourism_annual = (
    tourism_long[
        (tourism_long['month'] == 'TOTAL') &
        (tourism_long['indic_to'] == 'STY')
    ]
    [['GEO', 'TIME_PERIOD', 'c_resid', 'bookings']]
    .dropna(subset=['bookings'])
)
tourism_annual['TIME_PERIOD'] = tourism_annual['TIME_PERIOD'].astype(str)

print("tour_ce_omn12 – annual overnight stays (first rows):")
print(tourism_annual.head())
print("Shape:", tourism_annual.shape)


tour_ce_omn12 – annual overnight stays (first rows):
        GEO TIME_PERIOD c_resid  bookings
35183    AT        2018     DOM  118630.0
35184   AT1        2018     DOM   44518.0
35185  AT11        2018     DOM    4452.0
35186  AT12        2018     DOM    7217.0
35187  AT13        2018     DOM   32849.0
Shape: (8085, 4)


In [4]:
# --- Fetch and prepare ilc_lvho07_r (housing cost overburden rate, NUTS regions) ---
housing_raw = eurostat.get_data_df('ilc_lvho07_r')

print("ilc_lvho07_r columns:", housing_raw.columns.tolist())
housing_raw.head()


ilc_lvho07_r columns: ['freq', 'unit', 'geo\\TIME_PERIOD', '2021', '2022', '2023', '2024', '2025']


,freq,unit,geo\TIME_PERIOD,2021,2022,2023,2024,2025
0,A,PC,AL,3.3,2.8,2.1,NaN,NaN
1,A,PC,AT,6.1,7.4,6.0,6.3,NaN
2,A,PC,BE,7.6,7.5,7.8,6.7,6.7
3,A,PC,BE1,15.7,17.3,15.9,16.6,16.6
4,A,PC,BE10,15.7,17.3,15.9,16.6,16.6


In [5]:
# Identify the geo column (Eurostat names it "geo\TIME_PERIOD")
geo_col = [c for c in housing_raw.columns if 'geo' in c.lower()][0]

housing_raw = housing_raw.rename(columns={geo_col: "GEO"})

# Melt year columns to long format
id_cols = [c for c in housing_raw.columns if not c.replace('.', '').isdigit()]
year_cols = [c for c in housing_raw.columns if c not in id_cols]

housing_long = housing_raw.melt(
    id_vars=id_cols,
    value_vars=year_cols,
    var_name='TIME_PERIOD',
    value_name='housing_cost_overburden_rate'
)
housing_long['TIME_PERIOD'] = housing_long['TIME_PERIOD'].astype(str)
housing_long = housing_long.dropna(subset=['housing_cost_overburden_rate'])

print("ilc_lvho07_r – long format (first rows):")
print(housing_long.head())
print("Shape:", housing_long.shape)
print("GEO sample:", housing_long['GEO'].unique()[:10])


ilc_lvho07_r – long format (first rows):
  freq unit   GEO TIME_PERIOD  housing_cost_overburden_rate
0    A   PC    AL        2021                           3.3
1    A   PC    AT        2021                           6.1
2    A   PC    BE        2021                           7.6
3    A   PC   BE1        2021                          15.7
4    A   PC  BE10        2021                          15.7
Shape: (1459, 5)
GEO sample: ['AL' 'AT' 'BE' 'BE1' 'BE10' 'BE2' 'BE21' 'BE22' 'BE23' 'BE24']


In [6]:
# --- Merge on GEO (NUTS region code) and TIME_PERIOD ---
# Left-join tourism onto housing so we keep all tourism region-year pairs
# and enrich with the housing cost overburden rate where available.

merged = pd.merge(
    tourism_annual,
    housing_long[['GEO', 'TIME_PERIOD', 'housing_cost_overburden_rate']],
    on=['GEO', 'TIME_PERIOD'],
    how='left'
)

print("Merged dataset (first rows):")
print(merged.head(10))
print("\nShape:", merged.shape)
print("\nNull count in housing_cost_overburden_rate:", merged['housing_cost_overburden_rate'].isna().sum())
print("\nCoverage (non-null housing values):",
      f"{merged['housing_cost_overburden_rate'].notna().sum()} / {len(merged)}")
merged


Merged dataset (first rows):
    GEO TIME_PERIOD c_resid  bookings  housing_cost_overburden_rate
0    AT        2018     DOM  118630.0                           NaN
1   AT1        2018     DOM   44518.0                           NaN
2  AT11        2018     DOM    4452.0                           NaN
3  AT12        2018     DOM    7217.0                           NaN
4  AT13        2018     DOM   32849.0                           NaN
5   AT2        2018     DOM   33927.0                           NaN
6  AT21        2018     DOM   12237.0                           NaN
7  AT22        2018     DOM   21690.0                           NaN
8   AT3        2018     DOM   40185.0                           NaN
9  AT31        2018     DOM   10166.0                           NaN

Shape: (8085, 5)

Null count in housing_cost_overburden_rate: 4641

Coverage (non-null housing values): 3444 / 8085


,GEO,TIME_PERIOD,c_resid,bookings,housing_cost_overburden_rate
0,AT,2018,DOM,118630.0,NaN
1,AT1,2018,DOM,44518.0,NaN
2,AT11,2018,DOM,4452.0,NaN
3,AT12,2018,DOM,7217.0,NaN
4,AT13,2018,DOM,32849.0,NaN
...,...,...,...,...,...
8080,SK0,2024,TOTAL,484406.0,NaN
8081,SK01,2024,TOTAL,173514.0,5.2
8082,SK02,2024,TOTAL,53016.0,4.9
8083,SK03,2024,TOTAL,133435.0,6.7
